# Giai đoạn 8 — Lõi sản phẩm (`core.py` + `train_final.py`) dạng từng bước

Notebook này **viết lại toàn bộ `product_v3/core.py` và `train_final.py`** thành từng ô để đọc và debug:
mỗi hàm được định nghĩa ngay trong notebook (mã trích nguyên văn từ `core.py`), rồi chạy với dữ liệu thật
để xem kết quả trung gian.

| Bước | Hàm | Việc làm |
|---|---|---|
| 1 | `load_training_data` | Đọc 338 mẫu (V, H, S, M) |
| 2 | `make_features` | (V, H, S) → [V, H, log V, khô/ẩm, cát, mùn, vôi] |
| 3 | `make_pipeline`, `make_detector`, `p_mine_of` | **Hai mô hình**: phân loại 5 lớp (đặc trưng → chuẩn hoá → MLP 16-8) và phát hiện nhị phân (HistGradientBoosting) → P(có mìn) |
| 4 | `evaluate_cv` | CV lặp 5×3: accuracy, macro-F1, top-2 (phân loại); recall (phát hiện) |
| 5 | `estimate_threshold_cv` | Ngưỡng ưu tiên: chọn trên validation, đo trên test tách biệt |
| 6 | `train_and_save`, `load_bundle` | Huấn luyện trên toàn bộ dữ liệu → `models/model.pkl` |
| 7 | `validate_input`, `priority_level`, `predict_batch`, `summarize` | Dự đoán hàng loạt + xếp hạng ưu tiên |
| 8 | — | Kiểm tra: hàm trong notebook và `core.py` cho **cùng kết quả** |

> **App Streamlit dùng `core.py`, không dùng notebook.** Sửa lỗi ở đây xong thì chép sang `core.py`
> (ô kiểm tra cuối notebook sẽ báo nếu hai bên lệch nhau).

## 0. Thiết lập

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
import numpy as np, pandas as pd, joblib

PRODUCT = os.path.abspath("product_v3")          # notebook chạy từ thư mục Notebook/
if PRODUCT not in sys.path:
    sys.path.insert(0, PRODUCT)
import core                                       # bản .py mà app Streamlit dùng
print("core.py:", core.__file__)
print("Mô hình:", core.MODEL_NAME)

core.py: D:\2.Code\3.DATT_2026_Ver2\Notebook\product_v3\core.py
Mô hình: Phân loại: MLP(16,8) + log(V) + one-hot loại đất · Phát hiện: HistGradientBoosting nhị phân


In [2]:
# Hằng số (giống core.py)
FEATURES, CLASS_NAMES, SOIL_NAMES = core.FEATURES, core.CLASS_NAMES, core.SOIL_NAMES
DEFAULT_DATA, MODEL_PATH = core.DEFAULT_DATA, core.MODEL_PATH
MODEL_NAME, RANDOM_STATE, SAFETY_MARGIN = core.MODEL_NAME, core.RANDOM_STATE, core.SAFETY_MARGIN
print("Dữ liệu :", DEFAULT_DATA)
print("Mô hình :", MODEL_PATH)
print("Biên an toàn ngưỡng:", SAFETY_MARGIN)

Dữ liệu : D:\2.Code\3.DATT_2026_Ver2\Notebook\product_v3\data\raw\Mine_Dataset.xls
Mô hình : D:\2.Code\3.DATT_2026_Ver2\Notebook\product_v3\models\model.pkl
Biên an toàn ngưỡng: 0.8


## 1. Đọc dữ liệu huấn luyện

In [3]:
def load_training_data(data_path=DEFAULT_DATA):
    df = pd.read_excel(data_path) if data_path.lower().endswith((".xls", ".xlsx")) \
        else pd.read_csv(data_path)
    return df[FEATURES].values.astype(float), df["M"].values.astype(int)

In [4]:
X, y = load_training_data(DEFAULT_DATA)
print("X:", X.shape, "| y:", y.shape)
print(pd.Series(y).map(CLASS_NAMES).value_counts().to_string())
print("Phát hiện — có mìn / không mìn:", int((y != 1).sum()), "/", int((y == 1).sum()))
pd.DataFrame(X[:5], columns=FEATURES).assign(M=y[:5])

X: (338, 3) | y: (338,)
Không mìn                71
Mìn chống tăng           70
Mìn chống bộ binh        66
Mìn chống bộ binh bẫy    66
Mìn M14                  65
Phát hiện — có mìn / không mìn: 267 / 71


,V,H,S,M
0,0.338157,0.000000,0.0,1
1,0.320241,0.181818,0.0,1
2,0.287009,0.272727,0.0,1
3,0.256284,0.454545,0.0,1
4,0.262840,0.545455,0.0,1


## 2. Đặc trưng theo miền — `make_features`

- **log V**: dị thường từ giảm theo luật lũy thừa của khoảng cách, nên log làm quan hệ gần tuyến tính hơn.
- **S tách thành độ ẩm + vật liệu (one-hot)**: S chỉ là mã của 6 loại đất. Dùng thẳng S như một số
  sẽ áp đặt thứ tự giả ("khô-vôi" nằm giữa "khô-mùn" và "ẩm-cát").
- Hàm biến đổi **từng dòng độc lập**, không học tham số từ dữ liệu, nên không gây rò rỉ khi dùng trong CV.

In [5]:
def make_features(X):
    """(V, H, S) -> [V, H, log V, khô/ẩm, cát, mùn, vôi]. Biến đổi từng dòng, không học
    tham số từ dữ liệu nên không gây rò rỉ."""
    X = np.asarray(X, dtype=float)
    V, H, S = X[:, 0], X[:, 1], X[:, 2]
    idx = np.round(S * 5).astype(int)          # 0..5 = loại đất 1..6
    mat = idx % 3                              # 0 cát, 1 mùn, 2 vôi
    return np.column_stack([V, H, np.log(V + 1e-3), (idx >= 3).astype(float),
                            mat == 0, mat == 1, mat == 2]).astype(float)

In [6]:
# Xem đặc trưng của 1 mẫu mỗi loại đất
demo = np.array([[0.5, 0.3, s] for s in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]])
pd.DataFrame(make_features(demo), columns=["V", "H", "logV", "ẩm", "cát", "mùn", "vôi"],
             index=[SOIL_NAMES[s] for s in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]])

,V,H,logV,ẩm,cát,mùn,vôi
Khô & cát,0.5,0.3,-0.691149,0.0,1.0,0.0,0.0
Khô & mùn,0.5,0.3,-0.691149,0.0,0.0,1.0,0.0
Khô & vôi,0.5,0.3,-0.691149,0.0,0.0,0.0,1.0
Ẩm & cát,0.5,0.3,-0.691149,1.0,1.0,0.0,0.0
Ẩm & mùn,0.5,0.3,-0.691149,1.0,0.0,1.0,0.0
Ẩm & vôi,0.5,0.3,-0.691149,1.0,0.0,0.0,1.0


## 3. Hai mô hình — `make_pipeline` (phân loại), `make_detector` + `p_mine_of` (phát hiện)

Vì sao hai mô hình? MLP phân loại loại mìn tốt nhất (~72%) nhưng cho xác suất quá tự tin (nhiều giá trị = 1,0)
nên **xếp hạng** nguy cơ kém. Boosting nhị phân xếp hạng tốt hơn (ROC-AUC ~0,95 so với ~0,88), xem
`09_cai_thien_do_chinh_xac.ipynb` mục 4.

In [7]:
def make_pipeline():
    """Mô hình PHÂN LOẠI 5 lớp (loại mìn)."""
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler, FunctionTransformer
    from sklearn.neural_network import MLPClassifier
    return Pipeline([
        ("features", FunctionTransformer(make_features)),
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=5000,
                              random_state=RANDOM_STATE)),
    ])


def make_detector():
    """Mô hình PHÁT HIỆN nhị phân (1 = có mìn) — nguồn của P(có mìn), xếp hạng và ngưỡng."""
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import HistGradientBoostingClassifier
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))])


def p_mine_of(detector, X):
    """P(có mìn) từ mô hình phát hiện nhị phân."""
    return detector.predict_proba(X)[:, list(detector.classes_).index(1)]

In [8]:
# Debug nhanh trên MỘT lần chia (chỉ để xem; con số chính thức ở bước 4 dùng CV)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
m = make_pipeline().fit(Xtr, ytr)
pred = m.predict(Xte)
print(classification_report(yte, pred, target_names=[CLASS_NAMES[c] for c in sorted(CLASS_NAMES)], digits=3))
pd.DataFrame(confusion_matrix(yte, pred), index=[f"thật: {CLASS_NAMES[c]}" for c in sorted(CLASS_NAMES)],
             columns=[f"đoán: {c}" for c in sorted(CLASS_NAMES)])

                       precision    recall  f1-score   support

            Không mìn      0.773     0.944     0.850        18
       Mìn chống tăng      0.857     1.000     0.923        18
    Mìn chống bộ binh      0.769     0.588     0.667        17
Mìn chống bộ binh bẫy      0.727     0.500     0.593        16
              Mìn M14      0.556     0.625     0.588        16

             accuracy                          0.741        85
            macro avg      0.736     0.732     0.724        85
         weighted avg      0.740     0.741     0.731        85



,đoán: 1,đoán: 2,đoán: 3,đoán: 4,đoán: 5
thật: Không mìn,17,0,0,0,1
thật: Mìn chống tăng,0,18,0,0,0
thật: Mìn chống bộ binh,2,0,10,1,4
thật: Mìn chống bộ binh bẫy,0,3,2,8,3
thật: Mìn M14,3,0,1,2,10


In [9]:
d = make_detector().fit(Xtr, (ytr != 1).astype(int))
from sklearn.metrics import roc_auc_score
print("ROC-AUC phát hiện (1 lần chia):", round(roc_auc_score((yte != 1).astype(int), p_mine_of(d, Xte)), 3))
print("P(có mìn) của 8 mẫu test đầu:", np.round(p_mine_of(d, Xte[:8]), 3))
print("Nhãn thật                     :", yte[:8], "(1 = Không mìn)")

ROC-AUC phát hiện (1 lần chia): 0.962
P(có mìn) của 8 mẫu test đầu: [0.997 0.995 1.    1.    0.999 1.    0.726 1.   ]
Nhãn thật                     : [5 4 2 4 4 4 4 4] (1 = Không mìn)


## 4. Đánh giá CV — `evaluate_cv`

CV lặp 5 fold × 3 lần. Mỗi fold fit hai mô hình mới **chỉ trên train**: scaler, MLP và đặc trưng
đều nằm trong pipeline, nên không có thông tin nào từ test rò sang.

In [10]:
def evaluate_cv(X, y, n_splits=5, n_repeats=3, random_state=RANDOM_STATE):
    """CV lặp 5×3 trên đúng hai mô hình sản phẩm: accuracy, macro-F1, top-2 của mô hình
    phân loại 5 lớp; recall của mô hình phát hiện tại P(có mìn) ≥ 0.5."""
    from sklearn.model_selection import RepeatedStratifiedKFold
    from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score, recall_score

    yb = (y != 1).astype(int)
    cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                 random_state=random_state)
    acc, f1, top2, rec = [], [], [], []
    for tr, te in cv.split(X, y):
        m = make_pipeline().fit(X[tr], y[tr])
        P = m.predict_proba(X[te]); classes = list(m.classes_)
        pred = np.array(classes)[P.argmax(1)]
        acc.append(accuracy_score(y[te], pred))
        f1.append(f1_score(y[te], pred, average="macro"))
        top2.append(top_k_accuracy_score(y[te], P, k=2, labels=classes))
        d = make_detector().fit(X[tr], yb[tr])
        rec.append(recall_score(yb[te], (p_mine_of(d, X[te]) >= 0.5).astype(int)))
    pct = lambda v: round(float(np.mean(v)) * 100, 1)
    return {"cv_accuracy": pct(acc), "cv_accuracy_std": round(float(np.std(acc)) * 100, 1),
            "macro_f1_cv": pct(f1), "top2_accuracy_cv": pct(top2), "detect_recall_cv": pct(rec)}

In [11]:
cv_stats = evaluate_cv(X, y)
cv_stats

{'cv_accuracy': 71.8,
 'cv_accuracy_std': 4.6,
 'macro_f1_cv': 71.0,
 'top2_accuracy_cv': 88.3,
 'detect_recall_cv': 92.6}

## 5. Ngưỡng ưu tiên — `estimate_threshold_cv`

Mỗi fold: tách validation từ train, chọn **ngưỡng cao nhất vẫn bắt hết mìn trên validation**, rồi áp
ngưỡng × `SAFETY_MARGIN` (0,8), **đúng như sản phẩm**, lên test tách biệt.
Ngưỡng được tính **trên chính mô hình phát hiện** mà app dùng để tính P(có mìn).

In [12]:
def estimate_threshold_cv(X, y, n_splits=5, n_repeats=4, random_state=RANDOM_STATE):
    """
    Nhất quán Giai đoạn 6: mỗi fold fit mô hình phát hiện trên train, chọn ngưỡng
    recall=100% trên VALIDATION, rồi áp ĐÚNG ngưỡng sản phẩm (× SAFETY_MARGIN) lên
    TEST tách biệt. Trả về ngưỡng thận trọng + thống kê trung thực (recall thực trên
    test, tỷ lệ fold giữ được 100%, % không-mìn được xếp cuối, ROC-AUC).
    """
    from sklearn.model_selection import StratifiedKFold, train_test_split
    from sklearn.metrics import precision_recall_curve, confusion_matrix, roc_auc_score

    yb = (y != 1).astype(int)   # 1 = có mìn
    thr_list, test_recall, hold100, elim, auc = [], [], [], [], []
    for rep in range(n_repeats):
        skf = StratifiedKFold(n_splits, shuffle=True, random_state=random_state + rep)
        for tr, te in skf.split(X, y):
            Xt, Xv, yt, yv = train_test_split(X[tr], yb[tr], test_size=0.25,
                                              stratify=y[tr], random_state=random_state)
            m = make_detector().fit(Xt, yt)
            _, rec, thr = precision_recall_curve(yv, p_mine_of(m, Xv))
            idx = [i for i in range(len(thr)) if rec[i] >= 1.0]
            t = float(thr[max(idx)]) if idx else 0.0
            thr_list.append(t)
            p_te = p_mine_of(m, X[te])
            pred = (p_te >= t * SAFETY_MARGIN).astype(int)
            tn, fp, fn, tp = confusion_matrix(yb[te], pred, labels=[0, 1]).ravel()
            test_recall.append(tp / (tp + fn) if (tp + fn) else np.nan)
            hold100.append(1.0 if fn == 0 else 0.0)
            elim.append(tn / (tn + fp) if (tn + fp) else np.nan)
            auc.append(roc_auc_score(yb[te], p_te))
    safe_thr = float(np.nanmean(thr_list) * SAFETY_MARGIN)   # trung bình, hạ biên an toàn
    return {
        "safe_threshold": round(safe_thr, 4),
        "test_recall_mean": round(float(np.nanmean(test_recall)) * 100, 1),
        "test_recall_std": round(float(np.nanstd(test_recall)) * 100, 1),
        "hold_100_rate": round(float(np.mean(hold100)) * 100, 0),
        "deprioritizable_pct": round(float(np.nanmean(elim)) * 100, 1),
        "detect_roc_auc_cv": round(float(np.mean(auc)), 3),
    }

In [13]:
thr_stats = estimate_threshold_cv(X, y)
thr_stats

{'safe_threshold': 0.0972,
 'test_recall_mean': 99.4,
 'test_recall_std': 1.0,
 'hold_100_rate': 75.0,
 'deprioritizable_pct': 27.1,
 'detect_roc_auc_cv': 0.948}

## 6. Huấn luyện và lưu — `train_and_save` (tương đương `python train_final.py`)

> Bước lưu gọi `core.train_and_save` chứ không dùng bản định nghĩa trong notebook: pickle lưu **đường dẫn
> tới hàm** `make_features`. Nếu lưu từ notebook, đường dẫn sẽ là `__main__.make_features` và app sẽ không
> nạp được file `model.pkl`.

In [14]:
def train_and_save(data_path=DEFAULT_DATA, model_path=MODEL_PATH):
    """Đánh giá CV + ước lượng ngưỡng ưu tiên (CV) + fit trên toàn bộ dữ liệu. Lưu vào model_path."""
    X, y = load_training_data(data_path)
    cv_stats = evaluate_cv(X, y)
    thr_stats = estimate_threshold_cv(X, y)

    pipe = make_pipeline().fit(X, y)
    detector = make_detector().fit(X, (y != 1).astype(int))
    bundle = {
        "pipeline": pipe,        # phân loại 5 lớp
        "detector": detector,    # phát hiện nhị phân -> P(có mìn), xếp hạng, ngưỡng
        "meta": {
            "model_name": MODEL_NAME,
            "sklearn_version": _sklearn_version(),
            "features": FEATURES,
            "classes": CLASS_NAMES,
            **cv_stats,
            **thr_stats,
            "note": ("Hệ thống hỗ trợ sắp thứ tự ưu tiên. Ngưỡng KHÔNG đảm bảo "
                     "bắt 100% mìn trên dữ liệu mới — mọi tín hiệu VẪN phải đào "
                     "kiểm tra đầy đủ. Không dùng để bỏ qua bất kỳ tín hiệu nào."),
        },
    }
    os.makedirs(os.path.dirname(model_path), exist_ok=True)
    joblib.dump(bundle, model_path)
    return bundle["meta"]

In [15]:
meta = core.train_and_save(core.DEFAULT_DATA)      # = python train_final.py
pd.Series({k: v for k, v in meta.items() if k not in ("classes", "features", "note")})

model_name             Phân loại: MLP(16,8) + log(V) + one-hot loại đ...
sklearn_version                                                    1.5.2
cv_accuracy                                                         71.8
cv_accuracy_std                                                      4.6
macro_f1_cv                                                         71.0
top2_accuracy_cv                                                    88.3
detect_recall_cv                                                    92.6
safe_threshold                                                    0.0972
test_recall_mean                                                    99.4
test_recall_std                                                      1.0
hold_100_rate                                                       75.0
deprioritizable_pct                                                 27.1
detect_roc_auc_cv                                                  0.948
dtype: object

In [16]:
def _sklearn_version():
    import sklearn
    return sklearn.__version__


def _bundle_ok(b):
    """pkl hợp lệ nếu có đủ các trường meta của phiên bản hiện tại, đúng mô hình, và được
    tạo bằng ĐÚNG phiên bản scikit-learn đang chạy (lệch phiên bản -> sklearn chỉ cảnh báo
    'có thể sai kết quả', nên coi như không hợp lệ và tự train lại)."""
    need = {"safe_threshold", "cv_accuracy", "detect_recall_cv", "hold_100_rate",
            "model_name", "top2_accuracy_cv", "sklearn_version"}
    return (isinstance(b, dict) and "pipeline" in b and "detector" in b
            and need.issubset(b.get("meta", {}))
            and b["meta"]["model_name"] == MODEL_NAME
            and b["meta"]["sklearn_version"] == _sklearn_version())


def load_bundle(model_path=MODEL_PATH):
    """Nạp mô hình; TỰ PHỤC HỒI nếu pkl thiếu/hỏng/lệch phiên bản sklearn/mô hình cũ.
    Bình thường KHÔNG train lại — chỉ train một lần khi pkl không dùng được."""
    if "bundle" in core._cache:
        return core._cache["bundle"]
    import warnings
    b = None
    try:
        with warnings.catch_warnings():       # lệch phiên bản được xử lý bởi _bundle_ok
            warnings.simplefilter("ignore")
            b = joblib.load(model_path)
    except Exception as e:
        print("[core] Không nạp được model.pkl (%s)." % e)
    if not _bundle_ok(b):
        print("[core] model.pkl thiếu / mô hình cũ / lệch phiên bản scikit-learn "
              "(cần %s) -> tự huấn luyện lại một lần..." % _sklearn_version())
        if not os.path.exists(DEFAULT_DATA):
            raise RuntimeError("model.pkl thiếu/không hợp lệ và không tìm thấy dữ liệu "
                               "huấn luyện tại %s." % DEFAULT_DATA)
        core.train_and_save(DEFAULT_DATA, model_path)
        b = joblib.load(model_path)
    core._cache["bundle"] = b
    return b

In [17]:
core._cache.clear()
bundle = load_bundle(MODEL_PATH)
print("pkl hợp lệ:", _bundle_ok(bundle))
bundle["pipeline"]

pkl hợp lệ: True


Pipeline(steps=[('features',
                 FunctionTransformer(func=<function make_features at 0x0000012C0904CFE0>)),
                ('scaler', StandardScaler()),
                ('clf',
                 MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=5000,
                               random_state=42))])

## 7. Dự đoán hàng loạt — `validate_input`, `priority_level`, `predict_batch`, `summarize`

In [18]:
def validate_input(df_in):
    """Chuẩn hóa tên cột và kiểm tra V/H/S. Trả về DataFrame sạch hoặc ném ValueError
    kèm số dòng (theo Excel, dòng 1 là tiêu đề)."""
    df = df_in.copy()
    # chấp nhận tên cột có khoảng trắng / chữ thường: " v " -> "V"
    df = df.rename(columns={c: str(c).strip().upper() for c in df.columns
                            if str(c).strip().upper() in FEATURES})
    missing = [c for c in FEATURES if c not in df.columns]
    if missing:
        raise ValueError("Thiếu cột: %s. Cần có V, H, S." % missing)
    if len(df) == 0:
        raise ValueError("File không có dòng dữ liệu nào.")

    Xdf = df[FEATURES].apply(pd.to_numeric, errors="coerce")
    rows = lambda mask: ", ".join(str(i + 2) for i in np.flatnonzero(mask)[:10])
    bad = Xdf.isna().any(axis=1).values
    if bad.any():
        raise ValueError("Có %d dòng V/H/S trống hoặc không phải số (dòng %s)." % (bad.sum(), rows(bad)))
    out = ((Xdf < 0) | (Xdf > 1)).any(axis=1).values
    if out.any():
        raise ValueError("Có %d dòng V/H/S nằm ngoài [0, 1] (dòng %s). Dữ liệu phải được chuẩn hoá: "
                         "V = V_raw/10.6, H = H_raw/20, S ∈ {0.0, 0.2, …, 1.0}." % (out.sum(), rows(out)))
    off = (np.abs(Xdf["S"].values * 5 - np.round(Xdf["S"].values * 5)) > 1e-3)
    if off.any():
        raise ValueError("Có %d dòng S không thuộc 6 loại đất {0.0, 0.2, 0.4, 0.6, 0.8, 1.0} (dòng %s)."
                         % (off.sum(), rows(off)))
    df[FEATURES] = Xdf
    return df


def priority_level(p, safe_thr):
    """Mức ưu tiên đào theo P(có mìn)."""
    if p >= 0.5:            return "Rất cao"
    if p >= safe_thr:       return "Cao"
    if p >= safe_thr * 0.5: return "Trung bình"
    return "Thấp"

In [19]:
# Kiểm tra validate_input với các lỗi đầu vào thường gặp
sample = pd.read_excel(os.path.join(PRODUCT, "data", "sample_batch.xlsx"))
cases = {
    "hợp lệ":                sample,
    "tên cột viết thường":   sample.rename(columns={"V": " v "}),
    "ô trống":               sample.assign(V=sample.V.where(sample.index != 3)),
    "V chưa chuẩn hoá (V)":  sample.assign(V=sample.V * 10.6),
    "S sai":                 sample.assign(S=sample.S.where(sample.index != 0, 0.33)),
    "file rỗng":             sample.iloc[:0],
}
for name, d in cases.items():
    try:
        validate_input(d); print(f"{name:22s} -> OK")
    except ValueError as e:
        print(f"{name:22s} -> ValueError: {e}")

hợp lệ                 -> OK
tên cột viết thường    -> OK
ô trống                -> ValueError: Có 1 dòng V/H/S trống hoặc không phải số (dòng 5).
V chưa chuẩn hoá (V)   -> ValueError: Có 338 dòng V/H/S nằm ngoài [0, 1] (dòng 2, 3, 4, 5, 6, 7, 8, 9, 10, 11). Dữ liệu phải được chuẩn hoá: V = V_raw/10.6, H = H_raw/20, S ∈ {0.0, 0.2, …, 1.0}.
S sai                  -> ValueError: Có 1 dòng S không thuộc 6 loại đất {0.0, 0.2, 0.4, 0.6, 0.8, 1.0} (dòng 2).
file rỗng              -> ValueError: File không có dòng dữ liệu nào.


In [20]:
def predict_batch(df_in, model_path=MODEL_PATH):
    """DataFrame có V,H,S (tùy chọn id,x,y) -> thêm cột dự đoán + hạng ưu tiên."""
    bundle = load_bundle(model_path)
    pipe, detector, meta = bundle["pipeline"], bundle["detector"], bundle["meta"]
    safe_thr = meta["safe_threshold"]

    df = validate_input(df_in)
    X = df[FEATURES].values.astype(float)
    p_mine = p_mine_of(detector, X)                       # xếp hạng: mô hình phát hiện
    proba = pipe.predict_proba(X); classes = np.array(pipe.classes_)
    top = np.argsort(-proba, axis=1)                      # loại mìn: mô hình phân loại

    df["loai_du_doan"] = [meta["classes"][int(c)] for c in classes[top[:, 0]]]
    # top-2 đúng ~88% (CV): loại thứ hai giúp kỹ thuật viên chuẩn bị cho cả hai khả năng
    df["loai_kha_nang_2"] = [meta["classes"][int(c)] for c in classes[top[:, 1]]]
    df["xac_suat_co_min"] = np.round(p_mine, 4)
    df["muc_uu_tien"] = [priority_level(p, safe_thr) for p in p_mine]
    # cờ ưu tiên thấp: xếp CUỐI hàng đợi — KHÔNG có nghĩa bỏ qua (vẫn phải đào)
    df["uu_tien_thap"] = p_mine < safe_thr
    df["hang_uu_tien"] = df["xac_suat_co_min"].rank(
        ascending=False, method="first").astype(int)
    df = df.sort_values("hang_uu_tien").reset_index(drop=True)
    return df, meta


def summarize(df_pred):
    n = len(df_pred)
    by_level = df_pred["muc_uu_tien"].value_counts().to_dict()
    n_high = by_level.get("Rất cao", 0) + by_level.get("Cao", 0)
    n_low = int(df_pred["uu_tien_thap"].sum())
    return {
        "tong_tin_hieu": n,
        "uu_tien_cao": n_high,
        "uu_tien_thap": n_low,
        "ty_le_uu_tien_thap": round(100 * n_low / n, 1) if n else 0,
        "phan_bo_muc": by_level,
    }

In [21]:
pred, meta = predict_batch(sample)
print(summarize(pred))
pred.head(10)

{'tong_tin_hieu': 338, 'uu_tien_cao': 302, 'uu_tien_thap': 36, 'ty_le_uu_tien_thap': 10.7, 'phan_bo_muc': {'Rất cao': 264, 'Cao': 38, 'Thấp': 26, 'Trung bình': 10}}


,id,V,H,S,x,y,loai_du_doan,loai_kha_nang_2,xac_suat_co_min,muc_uu_tien,uu_tien_thap,hang_uu_tien
0,TH0051,0.906343,0.363636,0.0,107.003208,16.755796,Mìn chống tăng,Mìn chống bộ binh bẫy,1.0,Rất cao,False,1
1,TH0052,0.658609,0.545455,0.0,106.991818,16.742482,Mìn chống tăng,Mìn chống bộ binh bẫy,1.0,Rất cao,False,2
2,TH0053,0.610271,0.636364,0.0,107.007317,16.756820,Mìn chống tăng,Mìn chống bộ binh bẫy,1.0,Rất cao,False,3
3,TH0054,0.453172,0.818182,0.0,106.994986,16.757706,Mìn chống tăng,Mìn chống bộ binh bẫy,1.0,Rất cao,False,4
4,TH0055,0.442447,0.909091,0.0,107.008792,16.748884,Mìn chống tăng,Mìn chống bộ binh bẫy,1.0,Rất cao,False,5
5,TH0058,0.975830,0.272727,0.6,106.999799,16.733050,Mìn chống tăng,Mìn chống bộ binh,1.0,Rất cao,False,6
6,TH0059,0.815709,0.363636,0.6,106.987513,16.751887,Mìn chống tăng,Mìn chống bộ binh,1.0,Rất cao,False,7
7,TH0060,0.655588,0.545455,0.6,106.996861,16.752346,Mìn chống tăng,Mìn chống bộ binh,1.0,Rất cao,False,8
8,TH0061,0.628398,0.636364,0.6,107.000541,16.741345,Mìn chống tăng,Mìn chống bộ binh,1.0,Rất cao,False,9
9,TH0062,0.504531,0.818182,0.6,107.002728,16.757424,Mìn chống tăng,Mìn chống bộ binh,1.0,Rất cao,False,10


## 8. Kiểm tra: notebook và `core.py` cho cùng kết quả

Nếu ô này báo lỗi, nghĩa là code trong notebook và `core.py` đã lệch nhau; app vẫn chạy theo `core.py`.

In [22]:
pred_core, _ = core.predict_batch(sample)
pd.testing.assert_frame_equal(pred, pred_core)
assert np.allclose(make_features(X), core.make_features(X))
print("✓ Notebook và core.py cho kết quả giống hệt nhau trên", len(pred), "tín hiệu.")

✓ Notebook và core.py cho kết quả giống hệt nhau trên 338 tín hiệu.
